# Hybrid-architecture SAE — Qwen3.5-4B on Kaggle Free

**First public SAE on a Gated Delta Network. ~4 hours on 2×T4.**

---

This is **Tier 2** of the `openinterp.org/train` ladder:

| Tier | Hardware | Model | Tokens | Wall time | Cost |
|------|----------|-------|--------|-----------|------|
| 1 — Hobbyist | Colab T4 | Gemma-2-2B | 20M | ~30 min | $0 |
| **2 — Explorer (this)** | **Kaggle 2×T4** | **Qwen3.5-4B** | **150M** | **~4–5 h** | **$0** |
| 3 — Paper-grade | Vast.ai B200 | Qwen3.5-9B / Gemma-4 | 1B+ | ~22 h | ~$30 |

Qwen3.5-4B uses a **hybrid Gated Delta Network + attention** architecture. Training an SAE on its residual stream is still unmapped territory — this notebook is the cheapest path to playing there.

**Assumption**: you have completed Tier 1 (Gemma hobbyist) and know what TopK / AuxK / L0 / variance-explained mean.

**Kaggle quota**: Free tier gives 30h/week of accelerator time. This notebook consumes ~4–5 h of that budget.


In [ ]:
# Cell 2 — Install transformers from source (Qwen3.5 support is not in any stable release yet)
# ORDER MATTERS: stable torch already on Kaggle → causal-conv1d (needs torch 2.9+) → fla → transformers from source.
# If causal-conv1d wheel doesn't match Kaggle's torch, we skip it — fla has a Python fallback.

import sys, subprocess, shutil
from pathlib import Path

def pip(*a):
    return subprocess.run([sys.executable, '-m', 'pip', *a], check=False)

try:
    import transformers
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    has_qwen = 'qwen3_5' in CONFIG_MAPPING_NAMES
except Exception:
    has_qwen = False

print(f'qwen3_5 registered in transformers? {has_qwen}')

if not has_qwen:
    pip('install', '-q',
        'accelerate', 'datasets', 'huggingface_hub==1.5.0',
        'safetensors', 'einops', 'sentencepiece', 'tokenizers', 'protobuf')
    pip('uninstall', '-y', '-q', 'transformers', 'causal-conv1d')

    SRC = '/kaggle/working/transformers_src'
    if Path(SRC).exists():
        shutil.rmtree(SRC)
    subprocess.run(
        ['git', 'clone', '--quiet', '--depth=1',
         'https://github.com/huggingface/transformers.git', SRC],
        check=True)

    pip('install', '-q', '--force-reinstall', '--no-deps', '--no-cache-dir', SRC)
    pip('install', '-q', '--no-cache-dir', 'flash-linear-attention')
    # causal-conv1d is best-effort. If Kaggle's torch doesn't have a matching wheel,
    # fla falls back to a pure-python path that is ~2× slower for GDN layers but still usable.
    pip('install', '-q', '--no-cache-dir', 'causal-conv1d')

    print('\n*** RESTART RUNTIME NOW (Run → Restart Session), then rerun this cell ***')
else:
    print('Environment ready — transformers knows about qwen3_5. Continue.')


## Config — 150M tokens in ~4h on 2×T4

Ratios chosen for T4 (16 GB each, no bf16 tensor cores but bf16 works via emulation):

- `N_FEATURES = 40960` = 16× expansion over `D_MODEL = 2560`
- `K_TOPK = 128` — ~0.3% active features
- `K_AUX = 1280` (= d/2) with `ALPHA_AUX = 1/32` — dead-feature rescue
- `DEAD_TOKENS = 10M` — a feature is "dead" if unused for this many tokens
- `FWD_BATCH = 2 × SEQ_LEN 1024` on GPU0 for activations; SAE lives on GPU1

**150M tokens / (FWD_BATCH 2 × SEQ_LEN 1024) = ~73k forward passes ≈ 260 min at ~4.7 passes/sec on T4 = ~4.3 h.**


In [ ]:
# Cell 4 — Config
MODEL_ID       = 'Qwen/Qwen3.5-4B'
LAYER          = 18                 # mid-stack. Qwen3.5-4B has 36 layers.
D_MODEL        = 2560
N_FEATURES     = 40960              # 16x expansion
K_TOPK         = 128
K_AUX          = 1280               # d/2
ALPHA_AUX      = 1/32
DEAD_TOKENS    = 10_000_000
TOKEN_BUDGET   = 150_000_000
SEQ_LEN        = 1024
FWD_BATCH      = 2                  # batch of sequences per model forward
BATCH_SIZE     = 4096               # SAE optimizer batch (token-level)
LR_PEAK        = 2e-4
LR_FLOOR       = 6e-5
WARMUP_STEPS   = 3000
CKPT_EVERY_TOK = 10_000_000         # HF checkpoint cadence (kernel-kill safe)

HF_USERNAME    = 'YOUR_HF_USERNAME'  # <-- EDIT
HF_REPO        = f'{HF_USERNAME}/qwen35-4b-sae-L18-kaggle'

print(f'Target: {TOKEN_BUDGET/1e6:.0f}M tokens into L{LAYER} SAE of {N_FEATURES} features, k={K_TOPK}')
print(f'Checkpointing to hf://{HF_REPO} every {CKPT_EVERY_TOK/1e6:.0f}M tokens')


## Auth — Kaggle Secrets

Add your HF token as a Kaggle Secret named `HF_TOKEN` (Add-ons → Secrets → Add Secret). A write-scoped token is required for checkpoint uploads.


In [ ]:
# Cell 6 — HF auth via Kaggle Secrets
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, HfApi, create_repo

hf_token = UserSecretsClient().get_secret('HF_TOKEN')
login(token=hf_token)
api = HfApi(token=hf_token)

try:
    create_repo(HF_REPO, repo_type='model', exist_ok=True, token=hf_token)
    print(f'Repo ready: https://huggingface.co/{HF_REPO}')
except Exception as e:
    print(f'repo warn: {e}')


## Load Qwen3.5-4B

We use `AutoModelForImageTextToText` deliberately — Qwen3.5 is registered as a multimodal-capable class in the config even though the 4B we use is text-only. This loader correctly handles the hybrid GDN + attention path, whereas `AutoModelForCausalLM` sometimes strips the `model.language_model` prefix and leaves the GDN state modules unbound.

`device_map='auto'` will place the model on GPU0 (and spill to GPU1 if needed); we keep GPU1 mostly reserved for the SAE and its optimizer.


In [ ]:
# Cell 8 — Model load
import torch
from transformers import AutoTokenizer, AutoModelForImageTextToText

print(f'torch {torch.__version__} — {torch.cuda.device_count()} GPU(s)')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  [{i}] {p.name}  {p.total_memory/1e9:.1f} GB')

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,             # NEVER torch_dtype=
    device_map='auto',
    attn_implementation='sdpa',       # NEVER flash-attn on T4
    trust_remote_code=True,
)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

torch.cuda.empty_cache()
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f'  GPU{i} after model load: {(total-free)/1e9:.1f} / {total/1e9:.1f} GB used')

# Quick class sanity check
print(f'model class: {model.__class__.__name__}')
print(f'config model_type: {model.config.model_type}')


## Residual capture via `output_hidden_states`

Qwen3.5 multimodal wrappers hide `model.layers` behind inconsistent paths (sometimes `model.model.layers`, sometimes `model.language_model.layers`). We avoid the whole mess by asking for `output_hidden_states=True`.

`hidden_states[0]` is the embedding output. Decoder block `N` writes to `hidden_states[N+1]`. So for `LAYER = 18` we grab `hidden_states[19]`.

### Corpus mix
50 % **FineWeb-Edu** (broad web, high quality) + 50 % **OpenThoughts-114k** (reasoning traces) — reasoning bias is the point on a hybrid-arch model.


In [ ]:
# Cell 10 — Streaming dataset mix + hidden-state capture generator
import itertools, random
from datasets import load_dataset

LAYER_INDEX = LAYER + 1   # +1 because hidden_states[0] is embeddings

def open_streams(seed=0):
    # FineWeb-Edu (sample-10BT is the smallest canonical subset, fine for streaming)
    try:
        fw = load_dataset(
            'HuggingFaceFW/fineweb-edu',
            name='sample-10BT',
            split='train',
            streaming=True,
        ).shuffle(seed=seed, buffer_size=2000)
    except Exception as e:
        print(f'fineweb-edu sample-10BT failed ({e}); falling back to CC-MAIN-2024-10')
        fw = load_dataset(
            'HuggingFaceFW/fineweb-edu',
            name='CC-MAIN-2024-10',
            split='train',
            streaming=True,
        ).shuffle(seed=seed, buffer_size=2000)

    # OpenThoughts for reasoning mix. Schema varies by release — we try a couple.
    ot = None
    for name, field in [
        ('open-thoughts/OpenThoughts-114k', 'conversations'),
        ('open-thoughts/OpenThoughts-114k', 'system'),
    ]:
        try:
            ot = load_dataset(name, split='train', streaming=True).shuffle(seed=seed+1, buffer_size=2000)
            break
        except Exception as e:
            print(f'OT load {name} failed: {e}')
    if ot is None:
        print('WARN: no OpenThoughts, running 100% FineWeb-Edu')
    return fw, ot


def extract_text(sample):
    if 'text' in sample and isinstance(sample['text'], str):
        return sample['text']
    if 'conversations' in sample and isinstance(sample['conversations'], list):
        return '\n\n'.join(
            (m.get('value') or m.get('content') or '')
            for m in sample['conversations'] if isinstance(m, dict)
        )
    for k in ('content', 'output', 'response', 'answer'):
        if k in sample and isinstance(sample[k], str):
            return sample[k]
    return ''


def text_stream(seed=0):
    rng = random.Random(seed)
    fw, ot = open_streams(seed=seed)
    fw_it, ot_it = iter(fw), (iter(ot) if ot is not None else None)
    while True:
        src = fw_it if (ot_it is None or rng.random() < 0.5) else ot_it
        try:
            sample = next(src)
        except StopIteration:
            fw, ot = open_streams(seed=seed + rng.randint(1, 1_000_000))
            fw_it = iter(fw)
            ot_it = iter(ot) if ot is not None else None
            continue
        txt = extract_text(sample)
        if txt and len(txt) > 64:
            yield txt


def tokenize_pack(text_iter, seq_len=SEQ_LEN):
    """Pack short docs end-to-end into fixed-length sequences."""
    buf = []
    for txt in text_iter:
        ids = tok(txt, add_special_tokens=False).input_ids
        buf.extend(ids)
        buf.append(tok.eos_token_id or 0)
        while len(buf) >= seq_len:
            yield buf[:seq_len]
            buf = buf[seq_len:]


def batched(iterable, n):
    it = iter(iterable)
    while True:
        chunk = list(itertools.islice(it, n))
        if not chunk:
            return
        yield chunk


@torch.no_grad()
def activation_stream(seed=0):
    """Yields (FWD_BATCH*SEQ_LEN, D_MODEL) fp32 residual tensors on GPU1 (or GPU0 if single)."""
    dev_model = next(model.parameters()).device
    dev_sae   = torch.device('cuda:1') if torch.cuda.device_count() > 1 else dev_model
    seqs = tokenize_pack(text_stream(seed=seed))
    for chunk in batched(seqs, FWD_BATCH):
        ids = torch.tensor(chunk, dtype=torch.long, device=dev_model)
        out = model(input_ids=ids, output_hidden_states=True, use_cache=False)
        h = out.hidden_states[LAYER_INDEX]            # (B, T, D)
        h = h.reshape(-1, h.shape[-1]).to(dev_sae, dtype=torch.float32)
        del out
        yield h


# Smoke test — one batch
it = activation_stream(seed=123)
sample_batch = next(it)
print(f'sample activation batch: {tuple(sample_batch.shape)}  dtype={sample_batch.dtype}  dev={sample_batch.device}')
print(f'mean={sample_batch.mean().item():+.4f}  std={sample_batch.std().item():.4f}  '
      f'abs_max={sample_batch.abs().max().item():.2f}')
del it, sample_batch
torch.cuda.empty_cache()


## TopK SAE + AuxK

Standard OpenAI-style TopK SAE:

- Encoder: `Linear(d → n) → bias → TopK(k)` — everything else is zeroed.
- Decoder: `Linear(n → d)` with unit-norm columns (re-normalized after each step).
- AuxK: among features that haven't fired in `DEAD_TOKENS`, take the top `K_AUX` pre-activations and use them to explain the residual error. Gradient flows back — this is how dead features come back to life.
- Geometric-median initialization for `b_dec` (Weiszfeld) to put the decoder bias near the data mean.

SAE weights stay in **fp32** even though the model is bf16 — the stability cost is cheap and T4 tensor cores don't care about bf16.


In [ ]:
# Cell 12 — TopK SAE with AuxK
import torch
import torch.nn as nn
import torch.nn.functional as F

class TopKSAE(nn.Module):
    def __init__(self, d=D_MODEL, n=N_FEATURES, k=K_TOPK, k_aux=K_AUX, dead_tokens=DEAD_TOKENS):
        super().__init__()
        self.d, self.n, self.k, self.k_aux = d, n, k, k_aux
        self.dead_tokens = dead_tokens

        self.W_enc = nn.Parameter(torch.empty(d, n))
        self.b_enc = nn.Parameter(torch.zeros(n))
        self.W_dec = nn.Parameter(torch.empty(n, d))
        self.b_dec = nn.Parameter(torch.zeros(d))

        # Kaiming-ish init then force unit-norm decoder columns
        nn.init.kaiming_uniform_(self.W_enc, a=5**0.5)
        with torch.no_grad():
            self.W_dec.copy_(self.W_enc.t().contiguous())
            self.renorm_decoder()

        # Track: tokens-since-last-fire per feature
        self.register_buffer('last_fired', torch.zeros(n, dtype=torch.long))
        self.register_buffer('tokens_seen', torch.zeros(1, dtype=torch.long))

    @torch.no_grad()
    def renorm_decoder(self):
        nrm = self.W_dec.norm(dim=1, keepdim=True).clamp_min(1e-8)
        self.W_dec.div_(nrm)

    @torch.no_grad()
    def set_b_dec_geomedian(self, samples, iters=50, eps=1e-5):
        """Weiszfeld iteration. samples: (N, d) fp32."""
        x = samples.to(self.b_dec.device)
        mu = x.mean(0)
        for _ in range(iters):
            d = (x - mu).norm(dim=1).clamp_min(eps)
            w = 1.0 / d
            mu_new = (w[:, None] * x).sum(0) / w.sum()
            if (mu_new - mu).norm() < eps:
                break
            mu = mu_new
        self.b_dec.copy_(mu)

    def encode_pre(self, x):
        return (x - self.b_dec) @ self.W_enc + self.b_enc

    def forward(self, x):
        """x: (B, d)  → (recon, aux_recon, z, topk_idx, pre)"""
        pre = self.encode_pre(x)                             # (B, n)
        topk_val, topk_idx = pre.topk(self.k, dim=-1)
        z = torch.zeros_like(pre)
        z.scatter_(-1, topk_idx, F.relu(topk_val))
        recon = z @ self.W_dec + self.b_dec

        # AuxK: among dead features only
        aux_recon = None
        if self.training and self.k_aux > 0:
            dead_mask = (self.last_fired >= self.dead_tokens)       # (n,)
            n_dead = int(dead_mask.sum().item())
            if n_dead > 0:
                k_aux_eff = min(self.k_aux, n_dead)
                pre_dead = pre.masked_fill(~dead_mask, float('-inf'))
                aux_val, aux_idx = pre_dead.topk(k_aux_eff, dim=-1)
                z_aux = torch.zeros_like(pre)
                z_aux.scatter_(-1, aux_idx, F.relu(aux_val))
                aux_recon = z_aux @ self.W_dec

        return recon, aux_recon, z, topk_idx, pre

    @torch.no_grad()
    def update_fire_counter(self, topk_idx, batch_tokens):
        self.last_fired += batch_tokens
        # Features fired this batch: unique indices
        fired = torch.unique(topk_idx)
        self.last_fired[fired] = 0
        self.tokens_seen += batch_tokens

    @torch.no_grad()
    def dead_count(self):
        return int((self.last_fired >= self.dead_tokens).sum().item())


print('TopKSAE defined.')


## Initialize + resume from HF checkpoint if one exists

Kaggle kernels can be killed (idle timeout, preemption, lost connection). To survive that:
1. We upload `sae_L18_resume.pt` (weights + optimizer + scheduler + step + tokens) every `CKPT_EVERY_TOK`.
2. On (re)start, we try to download it and pick up where we left off.
3. On fresh start, we initialize `b_dec` with a geometric median over ~64 k activations.


In [ ]:
# Cell 14 — Instantiate SAE, optimizer, scheduler; try resume
import math, os, json
from huggingface_hub import hf_hub_download

dev_sae = torch.device('cuda:1') if torch.cuda.device_count() > 1 else torch.device('cuda:0')
print(f'SAE device: {dev_sae}')

sae = TopKSAE().to(dev_sae, dtype=torch.float32)

optim = torch.optim.Adam(sae.parameters(), lr=LR_PEAK, betas=(0.9, 0.999), eps=1e-8)

TOTAL_STEPS = max(1, TOKEN_BUDGET // BATCH_SIZE)

def lr_at(step):
    if step < WARMUP_STEPS:
        return LR_PEAK * step / max(1, WARMUP_STEPS)
    # cosine from LR_PEAK down to LR_FLOOR over the remaining steps
    prog = (step - WARMUP_STEPS) / max(1, TOTAL_STEPS - WARMUP_STEPS)
    prog = min(1.0, max(0.0, prog))
    return LR_FLOOR + 0.5 * (LR_PEAK - LR_FLOOR) * (1 + math.cos(math.pi * prog))

state = {'step': 0, 'tokens_seen': 0}

resumed = False
try:
    local = hf_hub_download(repo_id=HF_REPO, filename='sae_L18_resume.pt', token=hf_token)
    ckpt = torch.load(local, map_location='cpu', weights_only=False)
    sae.load_state_dict(ckpt['sae'])
    optim.load_state_dict(ckpt['optim'])
    state['step'] = int(ckpt.get('step', 0))
    state['tokens_seen'] = int(ckpt.get('tokens_seen', 0))
    sae.to(dev_sae, dtype=torch.float32)
    print(f'Resumed from step {state["step"]}, tokens_seen {state["tokens_seen"]/1e6:.1f}M')
    resumed = True
except Exception as e:
    print(f'No resume file on HF ({e}); fresh init with geometric median.')

if not resumed:
    # Warm up b_dec with geometric median over ~64k real activations
    geo_batches = []
    n_needed = 65536
    n_have = 0
    it = activation_stream(seed=42)
    while n_have < n_needed:
        b = next(it)
        geo_batches.append(b)
        n_have += b.shape[0]
    gm_samples = torch.cat(geo_batches, dim=0)[:n_needed]
    sae.set_b_dec_geomedian(gm_samples, iters=50)
    print(f'b_dec initialised via geom-median on {n_needed} tokens — '
          f'norm={sae.b_dec.norm().item():.3f}')
    del geo_batches, gm_samples, it
    torch.cuda.empty_cache()


## Training loop

We pull activation batches of `FWD_BATCH * SEQ_LEN = 2048` tokens from the model, chop them into `BATCH_SIZE = 4096`-token SAE optimizer steps (so one model forward feeds exactly half an optimizer step — we buffer).

Loss:

$$ \mathcal{L} = \underbrace{\lVert x - \hat{x} \rVert_2^2}_{\text{recon}} + \alpha_\text{aux} \underbrace{\lVert (x - \hat{x}) - \hat{x}_\text{aux} \rVert_2^2}_{\text{auxk on residual}} $$

Checkpoints go to HF every `CKPT_EVERY_TOK`. We log variance-explained, L0 (should equal K_TOPK), and dead-feature count.


In [ ]:
# Cell 16 — Training loop with HF checkpointing
import io, json, time
from safetensors.torch import save_file as save_safetensors
from huggingface_hub import upload_file
from tqdm.auto import tqdm

TMP = Path('/kaggle/working/ckpt'); TMP.mkdir(exist_ok=True)

def save_weights_safetensors(sae, path):
    sd = {k: v.detach().cpu().contiguous() for k, v in sae.state_dict().items()
          if not k.startswith(('last_fired', 'tokens_seen'))}
    save_safetensors(sd, str(path))

def save_resume(path, sae, optim, step, tokens_seen):
    torch.save({
        'sae': sae.state_dict(),
        'optim': optim.state_dict(),
        'step': step,
        'tokens_seen': tokens_seen,
    }, path)

def push_checkpoint(sae, optim, step, tokens_seen, is_final=False):
    w_path = TMP / 'sae_L18_latest.safetensors'
    r_path = TMP / 'sae_L18_resume.pt'
    cfg_path = TMP / 'cfg.json'
    save_weights_safetensors(sae, w_path)
    cfg = dict(
        model_id=MODEL_ID, layer=LAYER, d_model=D_MODEL, n_features=N_FEATURES,
        k_topk=K_TOPK, k_aux=K_AUX, alpha_aux=ALPHA_AUX,
        dead_tokens=DEAD_TOKENS, seq_len=SEQ_LEN,
        batch_size=BATCH_SIZE, token_budget=TOKEN_BUDGET,
        step=step, tokens_seen=tokens_seen, final=is_final,
    )
    cfg_path.write_text(json.dumps(cfg, indent=2))
    upload_file(path_or_fileobj=str(w_path), path_in_repo='sae_L18_latest.safetensors',
                repo_id=HF_REPO, token=hf_token)
    upload_file(path_or_fileobj=str(cfg_path), path_in_repo='cfg.json',
                repo_id=HF_REPO, token=hf_token)
    if not is_final:
        save_resume(r_path, sae, optim, step, tokens_seen)
        upload_file(path_or_fileobj=str(r_path), path_in_repo='sae_L18_resume.pt',
                    repo_id=HF_REPO, token=hf_token)

step = state['step']
tokens_seen = state['tokens_seen']
next_ckpt_at = ((tokens_seen // CKPT_EVERY_TOK) + 1) * CKPT_EVERY_TOK

pbar = tqdm(total=TOKEN_BUDGET, initial=tokens_seen, desc='training', unit='tok',
            unit_scale=True, smoothing=0.05)

acts_iter = activation_stream(seed=1000 + step)
acts_buf = torch.empty(0, D_MODEL, device=dev_sae, dtype=torch.float32)
running = {'recon': 0.0, 'aux': 0.0, 've': 0.0, 'n': 0}
t0 = time.time()

try:
    sae.train()
    while tokens_seen < TOKEN_BUDGET:
        # Fill buffer up to BATCH_SIZE
        while acts_buf.shape[0] < BATCH_SIZE:
            chunk = next(acts_iter)
            acts_buf = torch.cat([acts_buf, chunk], dim=0)
        x = acts_buf[:BATCH_SIZE]
        acts_buf = acts_buf[BATCH_SIZE:]

        # LR schedule
        lr = lr_at(step)
        for g in optim.param_groups:
            g['lr'] = lr

        recon, aux_recon, z, topk_idx, pre = sae(x)
        err = x - recon
        recon_loss = err.pow(2).mean()
        if aux_recon is not None:
            aux_err = err.detach() - aux_recon
            aux_loss = aux_err.pow(2).mean()
            loss = recon_loss + ALPHA_AUX * aux_loss
        else:
            aux_loss = torch.tensor(0.0, device=dev_sae)
            loss = recon_loss

        optim.zero_grad(set_to_none=True)
        loss.backward()
        # Zero the gradient along each decoder column's own direction (keep unit-norm)
        with torch.no_grad():
            if sae.W_dec.grad is not None:
                proj = (sae.W_dec.grad * sae.W_dec).sum(dim=1, keepdim=True)
                sae.W_dec.grad.sub_(proj * sae.W_dec)
        optim.step()
        with torch.no_grad():
            sae.renorm_decoder()
            sae.update_fire_counter(topk_idx, batch_tokens=BATCH_SIZE)

        # Metrics
        with torch.no_grad():
            var_x = x.var(unbiased=False).clamp_min(1e-8)
            ve = 1.0 - err.pow(2).mean() / var_x
            running['recon'] += recon_loss.item()
            running['aux']   += aux_loss.item() if aux_recon is not None else 0.0
            running['ve']    += ve.item()
            running['n']     += 1

        step += 1
        tokens_seen += BATCH_SIZE
        pbar.update(BATCH_SIZE)

        if step % 25 == 0:
            n = running['n']
            pbar.set_postfix(
                lr=f'{lr:.2e}',
                recon=f'{running["recon"]/n:.4f}',
                aux=f'{running["aux"]/n:.4f}',
                ve=f'{running["ve"]/n:.3f}',
                dead=sae.dead_count(),
                L0=K_TOPK,
            )
            running = {'recon': 0.0, 'aux': 0.0, 've': 0.0, 'n': 0}

        if tokens_seen >= next_ckpt_at:
            print(f'\n[ckpt @ {tokens_seen/1e6:.1f}M tokens, step {step}] uploading…')
            try:
                push_checkpoint(sae, optim, step, tokens_seen, is_final=False)
                print('[ckpt] done.')
            except Exception as e:
                print(f'[ckpt] upload failed, will retry next window: {e}')
            next_ckpt_at += CKPT_EVERY_TOK

finally:
    pbar.close()
    elapsed = time.time() - t0
    print(f'elapsed {elapsed/3600:.2f} h  —  tokens_seen {tokens_seen/1e6:.1f}M  —  step {step}')


## Final upload + held-out validation

1. Push `sae_L18_latest.safetensors` + `cfg.json` (final=true).
2. Delete `sae_L18_resume.pt` from the HF repo — it's only useful during training.
3. Run a clean forward on 500 k fresh tokens (different seed, no grad) and report VE / L0 / dead.


In [ ]:
# Cell 18 — Final checkpoint + held-out validation
from huggingface_hub import delete_file

print('Uploading final weights…')
try:
    push_checkpoint(sae, optim, step, tokens_seen, is_final=True)
    print('Final weights uploaded.')
except Exception as e:
    print(f'final upload error: {e}')

# Remove resume.pt — training is done
try:
    delete_file(path_in_repo='sae_L18_resume.pt', repo_id=HF_REPO, token=hf_token)
    print('resume.pt deleted.')
except Exception as e:
    print(f'resume delete warn: {e}')

# Validation on held-out (different seed, different stream offset)
print('\nValidation over 500k held-out tokens…')
sae.eval()
val_tokens = 500_000
val_seen = 0
val_recon_sse = 0.0
val_var_sum  = 0.0
L0_sum = 0.0
fired_ever = torch.zeros(N_FEATURES, dtype=torch.bool, device=dev_sae)

val_iter = activation_stream(seed=99999)
with torch.no_grad():
    while val_seen < val_tokens:
        chunk = next(val_iter)
        x = chunk[:min(chunk.shape[0], val_tokens - val_seen)]
        recon, _, z, topk_idx, _ = sae(x)
        err = x - recon
        val_recon_sse += err.pow(2).sum().item()
        val_var_sum   += (x - x.mean(0, keepdim=True)).pow(2).sum().item()
        L0_sum        += (z > 0).float().sum(dim=-1).sum().item()
        fired_ever[torch.unique(topk_idx)] = True
        val_seen += x.shape[0]

ve_val   = 1.0 - val_recon_sse / max(1e-9, val_var_sum)
L0_val   = L0_sum / max(1, val_seen)
dead_val = int((~fired_ever).sum().item())

report = dict(
    model_id=MODEL_ID, layer=LAYER,
    tokens_trained=tokens_seen, steps=step,
    val_tokens=val_seen,
    val_variance_explained=ve_val,
    val_L0=L0_val,
    val_dead_features=dead_val,
    val_dead_frac=dead_val / N_FEATURES,
    k_topk=K_TOPK, n_features=N_FEATURES,
)
print(json.dumps(report, indent=2))

rpath = TMP / 'val_report.json'
rpath.write_text(json.dumps(report, indent=2))
try:
    upload_file(path_or_fileobj=str(rpath), path_in_repo='val_report.json',
                repo_id=HF_REPO, token=hf_token)
    print(f'val_report.json uploaded to https://huggingface.co/{HF_REPO}')
except Exception as e:
    print(f'val report upload warn: {e}')

print('\nDone. Next tier: Vast.ai B200 for 1B tokens on Qwen3.5-9B or Gemma-4.')
